In [ ]:
import time
import numpy as np
import tensorflow.compat.v1 as tf
import scipy.sparse as sp
from torch_geometric.data import HeteroData
from utils import process
import os
from collections import defaultdict

"""
GCN (Graph Convolutional Network) Implementation for Cancer Subtype Classification

This file contains a complete GCN implementation that:
1. Processes heterogeneous graphs using meta-paths
2. Applies graph convolution layers for node classification
3. Compares performance with GAT and HAN models

Key components:
- GCN class with gcn_layer and inference methods
- Data preparation for homogeneous adjacency matrices
- Training loop with early stopping
- Meta-path exploration and evaluation
"""

# Disable TF v2 behavior
tf.disable_v2_behavior()

# Configure GPU
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2"
config = tf.ConfigProto()
config.gpu_options.allow_growth = True

# Set random seed for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.set_random_seed(RANDOM_SEED)

def to_numpy(x):
    """Convert tensor to numpy array"""
    try:
        import torch
        if isinstance(x, torch.Tensor):
            return x.cpu().numpy()
    except ImportError:
        pass
    return np.array(x)

def build_bipartite_adj(src_idx, dst_idx, shape_src, shape_dst):
    """Build a scipy sparse COO adjacency from src to dst"""
    data = np.ones(len(src_idx), dtype=np.float32)
    return sp.coo_matrix((data, (src_idx, dst_idx)), shape=(shape_src, shape_dst))

def build_meta_path_adj(data: HeteroData, meta_path: list):
    """Build adjacency among target nodes via sequence of relations in meta_path"""
    # First relation
    s0, r0, d0 = meta_path[0]
    e0 = to_numpy(data[s0, r0, d0].edge_index)
    A = build_bipartite_adj(e0[0], e0[1], data[s0].num_nodes, data[d0].num_nodes)
    curr = d0
    
    # Remaining relations
    for (src, rel, dst) in meta_path[1:]:
        assert curr == src, f"Meta-path mismatch: expected {curr}, got {src}"
        e = to_numpy(data[src, rel, dst].edge_index)
        B = build_bipartite_adj(e[0], e[1], data[src].num_nodes, data[dst].num_nodes)
        A = A.dot(B)
        curr = dst
    
    A = A.tocoo()
    if A.shape[0] == A.shape[1]:
        A.setdiag(0)
    A.eliminate_zeros()
    return A

def generate_meta_paths(data: HeteroData, max_length=3, start_end='case'):
    """Generate all meta-paths up to max_length hops that start and end at start_end node type"""
    edge_types = data.edge_types
    outgoing = defaultdict(list)
    
    for src, rel, dst in edge_types:
        outgoing[src].append((src, rel, dst))

    results = []
    
    def dfs(path, curr_type):
        if len(path) == max_length:
            if path[0][0] == path[-1][2] == start_end:
                results.append(path.copy())
            return
        
        for edge in outgoing.get(curr_type, []):
            path.append(edge)
            dfs(path, edge[2])
            path.pop()

    for edge in outgoing.get(start_end, []):
        dfs([edge], edge[2])
    
    return results

def prepare_data_from_hetero(data, target_node_type='case', meta_paths=None,
                           split_ratios=(0.7, 0.15, 0.15), random_state=42):
    """Prepare data from heterogeneous graph for GCN training"""
    
    # Features
    if not hasattr(data[target_node_type], 'x'):
        raise ValueError(f"Target node type '{target_node_type}' has no .x features")
    feat = to_numpy(data[target_node_type].x)
    N, ft_size = feat.shape
    
    # Labels
    if not hasattr(data[target_node_type], 'y'):
        raise ValueError(f"Target node type '{target_node_type}' has no .y labels")
    labels = to_numpy(data[target_node_type].y).astype(int).reshape(-1)
    num_classes = labels.max() + 1
    y_all = np.eye(num_classes, dtype=np.float32)[labels]
    
    # Masks
    has_train = hasattr(data[target_node_type], 'train_mask')
    has_val = hasattr(data[target_node_type], 'val_mask')
    has_test = hasattr(data[target_node_type], 'test_mask')
    
    if has_train and has_val and has_test:
        train_mask = to_numpy(data[target_node_type].train_mask).astype(bool)
        val_mask = to_numpy(data[target_node_type].val_mask).astype(bool)
        test_mask = to_numpy(data[target_node_type].test_mask).astype(bool)
    else:
        from sklearn.model_selection import StratifiedShuffleSplit
        idx = np.arange(N)
        s1 = StratifiedShuffleSplit(n_splits=1, test_size=1 - split_ratios[0], random_state=random_state)
        tr_idx, rest = next(s1.split(idx, labels))
        val_prop = split_ratios[1] / (split_ratios[1] + split_ratios[2])
        s2 = StratifiedShuffleSplit(n_splits=1, test_size=1 - val_prop, random_state=random_state)
        v_idx, te_idx_rel = next(s2.split(rest, labels[rest]))
        te_idx = rest[te_idx_rel]
        
        train_mask = np.zeros(N, bool)
        train_mask[tr_idx] = True
        val_mask = np.zeros(N, bool)
        val_mask[rest[v_idx]] = True
        test_mask = np.zeros(N, bool)
        test_mask[te_idx] = True
    
    # Build adjacency matrices for meta-paths
    if meta_paths is None:
        raise ValueError("Provide meta_paths list")
    
    # For GCN, we typically use a single adjacency matrix (can combine multiple meta-paths)
    if len(meta_paths) == 1:
        A = build_meta_path_adj(data, meta_paths[0])
    else:
        # Combine multiple meta-paths by summing their adjacencies
        A_combined = None
        for mp in meta_paths:
            A = build_meta_path_adj(data, mp)
            if A_combined is None:
                A_combined = A
            else:
                A_combined = A_combined + A
        A = A_combined
    
    # Convert to dense and normalize
    A_dense = A.toarray().astype(np.float32)
    
    # Add self-loops
    A_dense = A_dense + np.eye(N, dtype=np.float32)
    
    # Symmetric normalization: D^(-1/2) * A * D^(-1/2)
    rowsum = np.array(A_dense.sum(1))
    d_inv_sqrt = np.power(rowsum, -0.5).flatten()
    d_inv_sqrt[np.isinf(d_inv_sqrt)] = 0.
    d_mat_inv_sqrt = np.diag(d_inv_sqrt)
    A_normalized = d_mat_inv_sqrt.dot(A_dense).dot(d_mat_inv_sqrt)
    
    # Labels & masks
    y_tr = np.zeros_like(y_all)
    y_tr[train_mask] = y_all[train_mask]
    y_va = np.zeros_like(y_all)
    y_va[val_mask] = y_all[val_mask]
    y_te = np.zeros_like(y_all)
    y_te[test_mask] = y_all[test_mask]
    
    return (feat, A_normalized, y_tr, y_va, y_te,
            train_mask, val_mask, test_mask, N, ft_size, num_classes, y_all)

class GCN:
    """Simple GCN implementation - TensorFlow 1.x compatible"""
    
    @staticmethod
    def gcn_layer(inputs, adj_matrix, output_dim, activation=None, dropout_rate=0.0, is_training=None, name="gcn"):
        """Single GCN layer: A * X * W"""
        with tf.variable_scope(name, reuse=tf.AUTO_REUSE):
            # Get input dimension
            input_dim = inputs.get_shape().as_list()[-1]
            
            # Weight matrix
            W = tf.get_variable("weights", [input_dim, output_dim], 
                              initializer=tf.glorot_uniform_initializer())
            
            # Apply dropout only if is_training is provided and dropout_rate > 0
            if is_training is not None and dropout_rate > 0:
                inputs_dropped = tf.cond(
                    is_training,
                    lambda: tf.nn.dropout(inputs, keep_prob=1.0 - dropout_rate),
                    lambda: inputs
                )
            else:
                inputs_dropped = inputs
            
            # GCN operation: A * X * W
            support = tf.matmul(inputs_dropped, W)
            output = tf.matmul(adj_matrix, support)
            
            # Apply activation if specified
            if activation is not None:
                output = activation(output)
                
            return output
    
    @staticmethod
    def inference(inputs, adj_matrix, nb_classes, is_training=None, dropout_rate=0.5, 
                 hidden_dims=[16], activation=tf.nn.relu):
        """GCN inference with multiple layers"""
        
        h = inputs
        
        # Hidden layers
        for i, hidden_dim in enumerate(hidden_dims):
            h = GCN.gcn_layer(
                inputs=h, 
                adj_matrix=adj_matrix, 
                output_dim=hidden_dim, 
                activation=activation, 
                dropout_rate=dropout_rate, 
                is_training=is_training,
                name=f"gcn_layer_{i}"
            )
        
        # Output layer (no dropout, no activation)
        logits = GCN.gcn_layer(
            inputs=h, 
            adj_matrix=adj_matrix, 
            output_dim=nb_classes, 
            activation=None, 
            dropout_rate=0.0,
            is_training=is_training,
            name="gcn_output"
        )
        
        return logits
    
    @staticmethod
    def masked_softmax_cross_entropy(preds, labels, mask):
        """Softmax cross-entropy loss with masking"""
        loss = tf.nn.softmax_cross_entropy_with_logits_v2(logits=preds, labels=labels)
        mask = tf.cast(mask, dtype=tf.float32)
        mask /= tf.reduce_mean(mask)
        loss *= mask
        return tf.reduce_mean(loss)
    
    @staticmethod
    def masked_accuracy(preds, labels, mask):
        """Accuracy with masking"""
        correct_prediction = tf.equal(tf.argmax(preds, 1), tf.argmax(labels, 1))
        accuracy_all = tf.cast(correct_prediction, tf.float32)
        mask = tf.cast(mask, dtype=tf.float32)
        mask /= tf.reduce_mean(mask)
        accuracy_all *= mask
        return tf.reduce_mean(accuracy_all)
    
    @staticmethod
    def training(loss, learning_rate, l2_coef):
        """Training operation with L2 regularization"""
        # L2 regularization
        vars = tf.trainable_variables()
        lossL2 = tf.add_n([tf.nn.l2_loss(v) for v in vars if 'bias' not in v.name]) * l2_coef
        
        # Optimizer
        opt = tf.train.AdamOptimizer(learning_rate=learning_rate)
        
        # Training op
        train_op = opt.minimize(loss + lossL2)
        
        return train_op

def run_gcn_for_path(data: HeteroData, meta_path, verbose=True):
    """Run GCN for a single meta-path or meta-path combination"""
    
    # Reset graph for clean training
    tf.reset_default_graph()
    
    # Prepare data
    (feat, adj_matrix, y_tr, y_va, y_te,
     train_mask, val_mask, test_mask, N, ft_size, num_classes, _y_all) = \
        prepare_data_from_hetero(data, 'case', [meta_path])
    
    # Placeholders
    features_ph = tf.placeholder(tf.float32, [N, ft_size], name="features")
    adj_ph = tf.placeholder(tf.float32, [N, N], name="adjacency")
    labels_ph = tf.placeholder(tf.float32, [N, num_classes], name="labels")
    mask_ph = tf.placeholder(tf.bool, [N], name="mask")
    is_training_ph = tf.placeholder(tf.bool, name="is_training")
    
    # Model
    logits = GCN.inference(
        inputs=features_ph,
        adj_matrix=adj_ph,
        nb_classes=num_classes,
        is_training=is_training_ph,
        dropout_rate=0.5,
        hidden_dims=[16],
        activation=tf.nn.relu
    )
    
    # Loss and accuracy
    loss = GCN.masked_softmax_cross_entropy(logits, labels_ph, mask_ph)
    accuracy = GCN.masked_accuracy(logits, labels_ph, mask_ph)
    train_op = GCN.training(loss, learning_rate=0.01, l2_coef=5e-4)
    
    # Training
    saver = tf.train.Saver()
    init = tf.group(tf.global_variables_initializer(), tf.local_variables_initializer())
    
    with tf.Session(config=config) as sess:
        sess.run(init)
        
        best_val_acc = 0.0
        best_val_loss = np.inf
        patience = 0
        max_patience = 50
        
        for epoch in range(200):
            # Training step
            feed_dict_train = {
                features_ph: feat,
                adj_ph: adj_matrix,
                labels_ph: y_tr,
                mask_ph: train_mask,
                is_training_ph: True
            }
            
            _, train_loss, train_acc = sess.run([train_op, loss, accuracy], feed_dict_train)
            
            # Validation step
            feed_dict_val = {
                features_ph: feat,
                adj_ph: adj_matrix,
                labels_ph: y_va,
                mask_ph: val_mask,
                is_training_ph: False
            }
            
            val_loss, val_acc = sess.run([loss, accuracy], feed_dict_val)
            
            # Early stopping
            if val_acc > best_val_acc or (val_acc == best_val_acc and val_loss < best_val_loss):
                best_val_acc = val_acc
                best_val_loss = val_loss
                saver.save(sess, 'gcn_best.ckpt')
                patience = 0
            else:
                patience += 1
                if patience >= max_patience:
                    break
        
        # Test
        saver.restore(sess, 'gcn_best.ckpt')
        feed_dict_test = {
            features_ph: feat,
            adj_ph: adj_matrix,
            labels_ph: y_te,
            mask_ph: test_mask,
            is_training_ph: False
        }
        
        test_loss, test_acc = sess.run([loss, accuracy], feed_dict_test)
        
        return float(test_acc)

def run_gcn_experiment(hetero_data: HeteroData, max_path_len=3, top_k_paths=10):
    """Run GCN experiment on multiple meta-paths"""
    
    print("="*80)
    print("GCN CANCER SUBTYPE CLASSIFICATION EXPERIMENT")
    print("="*80)
    
    # Generate meta-paths
    all_paths = generate_meta_paths(hetero_data, max_path_len)
    print(f"→ Found {len(all_paths)} meta-paths of length <= {max_path_len}")
    
    if len(all_paths) == 0:
        print("No valid meta-paths found!")
        return None
    
    results = []
    
    # Test each meta-path
    for i, mp in enumerate(all_paths, 1):
        path_str = " -> ".join([f"{s}--{r}-->{d}" for s, r, d in mp])
        print(f"\n[{i}/{len(all_paths)}] Testing path: {path_str}")
        
        try:
            start_time = time.time()
            acc = run_gcn_for_path(hetero_data, mp)
            training_time = time.time() - start_time
            
            print(f"  ✓ Test accuracy: {acc:.4f} (trained in {training_time:.2f}s)")
            results.append((mp, acc, path_str))
            
        except Exception as e:
            print(f"  ✗ Skipped: {e}")
    
    if not results:
        print("No successful experiments!")
        return None
    
    # Sort by accuracy
    results.sort(key=lambda x: -x[1])
    
    print("\n" + "="*80)
    print("GCN EXPERIMENT RESULTS")
    print("="*80)
    
    print(f"\nTop {min(top_k_paths, len(results))} Meta-paths:")
    for i, (mp, acc, path_str) in enumerate(results[:top_k_paths], 1):
        print(f"{i:2d}. Accuracy: {acc:.4f} | Path: {path_str}")
    
    best_mp, best_acc, best_path_str = results[0]
    print(f"\n🏆 Best GCN result:")
    print(f"   Accuracy: {best_acc:.4f}")
    print(f"   Meta-path: {best_path_str}")
    
    return {
        'best_metapath': best_mp,
        'best_accuracy': best_acc,
        'best_path_string': best_path_str,
        'all_results': results,
        'model_type': 'GCN'
    }

def main(hetero_data, max_path_length=3, top_k=10):
    """Main function to run GCN experiment"""
    return run_gcn_experiment(hetero_data, max_path_length, top_k)
